# Error Analysis & Embeddings Visualization

Детальный анализ ошибок моделей и визуализация эмбеддингов:
- Confusion matrix по городам
- Анализ географических ошибок
- t-SNE визуализация эмбеддингов
- Примеры успешных и неуспешных предсказаний
- Heatmap ошибок на карте

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report
import torch
from PIL import Image
import folium
from folium.plugins import HeatMap

# Добавляем путь к модулям проекта
sys.path.append('../code')

from models import build_model
from dataset import GeoDataset, get_transforms
from metrics import haversine_distance

# Настройки визуализации
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

%matplotlib inline

## 1. Загрузка модели и предсказаний

In [ ]:
def load_model_and_predictions(checkpoint_path: str, device: str = 'cuda'):
    """
    Загружает обученную модель и генерирует предсказания на тестовом наборе.
    
    Args:
        checkpoint_path: Путь к checkpoint файлу
        device: 'cuda' или 'cpu'
    
    Returns:
        Tuple: (model, predictions, ground_truth, embeddings, class_names)
    """
    checkpoint_path = Path(checkpoint_path)
    
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Извлекаем параметры
    config = checkpoint.get('config', {})
    class_names = checkpoint.get('class_names', [])
    num_classes = len(class_names)
    
    # Создаем модель
    architecture = config.get('architecture', 'baseline')
    model = build_model(architecture, num_classes, pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Model loaded: {architecture} with {num_classes} classes")
    print(f"Classes: {class_names}")
    
    return model, config, class_names


def generate_predictions(model, test_loader, device='cuda'):
    """
    Генерирует предсказания и эмбеддинги для тестового набора.
    """
    all_preds = []
    all_labels = []
    all_coords = []
    all_embeddings = []
    
    with torch.no_grad():
        for images, labels, coords in test_loader:
            images = images.to(device)
            
            # Получаем предсказания
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            # Получаем эмбеддинги (features перед классификационным слоем)
            if hasattr(model, 'get_embeddings'):
                embeddings = model.get_embeddings(images)
            else:
                # Fallback - используем features из базового слоя
                embeddings = model.backbone(images)
                embeddings = torch.flatten(embeddings, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_coords.extend(coords.numpy())
            all_embeddings.extend(embeddings.cpu().numpy())
    
    return {
        'predictions': np.array(all_preds),
        'ground_truth': np.array(all_labels),
        'coordinates': np.array(all_coords),
        'embeddings': np.array(all_embeddings)
    }


# Загрузка модели (выберите нужную)
MODEL_CHECKPOINT = '../checkpoints/baseline/best_model.pth'  # Измените на нужный путь

try:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    model, config, class_names = load_model_and_predictions(MODEL_CHECKPOINT, device)
    
    # Загружаем тестовый датасет
    test_dataset = GeoDataset(
        manifest_path='../dataset/manifests/test.csv',
        transform=get_transforms('val'),
        class_names=class_names
    )
    
    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4
    )
    
    print(f"\nTest dataset: {len(test_dataset)} images")
    print("Generating predictions...")
    
    results = generate_predictions(model, test_loader, device)
    
    print(f"✅ Generated {len(results['predictions'])} predictions")
    
except FileNotFoundError as e:
    print(f"⚠️ Error: {e}")
    print("Please train a model first or adjust MODEL_CHECKPOINT path.")

## 2. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, normalize=True):
    """
    Визуализирует confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = 'Normalized Confusion Matrix'
        fmt = '.2f'
    else:
        title = 'Confusion Matrix'
        fmt = 'd'
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt=fmt,
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Proportion' if normalize else 'Count'}
    )
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Predicted City', fontsize=13, fontweight='bold')
    plt.ylabel('True City', fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig('../results/plots/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

if 'results' in locals():
    Path('../results/plots').mkdir(parents=True, exist_ok=True)
    plot_confusion_matrix(
        results['ground_truth'],
        results['predictions'],
        class_names,
        normalize=True
    )

## 3. Per-Class Performance

In [ ]:
def print_classification_report(y_true, y_pred, class_names):
    """
    Выводит детальный отчет по метрикам для каждого класса.
    """
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3,
        output_dict=True
    )
    
    # Конвертируем в DataFrame для красивого отображения
    report_df = pd.DataFrame(report).transpose()
    
    print("\n📊 Classification Report:\n")
    print(report_df.to_string())
    
    # Сохраняем в CSV
    report_df.to_csv('../results/classification_report.csv')
    print("\n✅ Saved to: results/classification_report.csv")
    
    return report_df

if 'results' in locals():
    report_df = print_classification_report(
        results['ground_truth'],
        results['predictions'],
        class_names
    )

## 4. Geographic Error Analysis

In [ ]:
def analyze_geographic_errors(predictions, ground_truth, coordinates, class_names):
    """
    Анализирует географические ошибки предсказаний.
    """
    # Вычисляем расстояния для неправильных предсказаний
    errors = predictions != ground_truth
    
    # Примерные центры городов (для демонстрации - замените на реальные)
    CITY_CENTERS = {
        'Kyiv': (50.4501, 30.5234),
        'Lviv': (49.8397, 24.0297),
        'Odesa': (46.4825, 30.7233),
        'Kharkiv': (49.9935, 36.2304),
        'Dnipro': (48.4647, 35.0462),
        'Warsaw': (52.2297, 21.0122),
        'Prague': (50.0755, 14.4378),
        'Budapest': (47.4979, 19.0402),
    }
    
    error_distances = []
    
    for i in range(len(predictions)):
        if errors[i]:
            true_city = class_names[ground_truth[i]]
            pred_city = class_names[predictions[i]]
            
            if true_city in CITY_CENTERS and pred_city in CITY_CENTERS:
                true_coords = CITY_CENTERS[true_city]
                pred_coords = CITY_CENTERS[pred_city]
                
                dist = haversine_distance(
                    np.array([true_coords]),
                    np.array([pred_coords])
                )[0]
                
                error_distances.append({
                    'true_city': true_city,
                    'pred_city': pred_city,
                    'distance_km': dist
                })
    
    error_df = pd.DataFrame(error_distances)
    
    if len(error_df) > 0:
        print("\n📍 Geographic Error Analysis:\n")
        print(f"Total misclassifications: {len(error_df)}")
        print(f"Mean error distance: {error_df['distance_km'].mean():.1f} km")
        print(f"Median error distance: {error_df['distance_km'].median():.1f} km")
        print(f"Max error distance: {error_df['distance_km'].max():.1f} km")
        
        # Топ-5 самых частых ошибок
        print("\nMost common misclassifications:")
        error_pairs = error_df.groupby(['true_city', 'pred_city']).size()
        print(error_pairs.sort_values(ascending=False).head(10))
        
        # Визуализация распределения ошибок
        plt.figure(figsize=(10, 6))
        plt.hist(error_df['distance_km'], bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel('Error Distance (km)', fontsize=12, fontweight='bold')
        plt.ylabel('Frequency', fontsize=12, fontweight='bold')
        plt.title('Distribution of Geographic Errors', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('../results/plots/error_distance_distribution.png', dpi=300)
        plt.show()
    
    return error_df

if 'results' in locals():
    error_df = analyze_geographic_errors(
        results['predictions'],
        results['ground_truth'],
        results['coordinates'],
        class_names
    )

## 5. t-SNE Embeddings Visualization

In [ ]:
def plot_tsne_embeddings(embeddings, labels, class_names, perplexity=30, n_iter=1000):
    """
    Визуализирует эмбеддинги с помощью t-SNE.
    
    Args:
        embeddings: Numpy array формы (N, D) - N эмбеддингов размерности D
        labels: Numpy array формы (N,) - метки классов
        class_names: Список названий классов
        perplexity: Параметр t-SNE (обычно 5-50)
        n_iter: Количество итераций t-SNE
    """
    print(f"\nRunning t-SNE on {len(embeddings)} embeddings...")
    print(f"Embedding dimensionality: {embeddings.shape[1]}")
    
    # Применяем t-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        n_iter=n_iter,
        random_state=42,
        verbose=1
    )
    
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # Визуализация
    plt.figure(figsize=(14, 10))
    
    # Цветовая палитра
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    
    for i, city in enumerate(class_names):
        mask = labels == i
        plt.scatter(
            embeddings_2d[mask, 0],
            embeddings_2d[mask, 1],
            c=[colors[i]],
            label=city,
            alpha=0.6,
            s=30,
            edgecolors='black',
            linewidths=0.5
        )
    
    plt.xlabel('t-SNE Dimension 1', fontsize=13, fontweight='bold')
    plt.ylabel('t-SNE Dimension 2', fontsize=13, fontweight='bold')
    plt.title('t-SNE Visualization of Image Embeddings by City', fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='best', fontsize=11, framealpha=0.9)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/plots/tsne_embeddings.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ t-SNE visualization complete!")
    
    return embeddings_2d

if 'results' in locals():
    # Можем взять подмножество для ускорения (если датасет большой)
    n_samples = min(2000, len(results['embeddings']))
    indices = np.random.choice(len(results['embeddings']), n_samples, replace=False)
    
    embeddings_2d = plot_tsne_embeddings(
        results['embeddings'][indices],
        results['ground_truth'][indices],
        class_names,
        perplexity=30
    )

## 6. Error Heatmap on Map

In [ ]:
def create_error_heatmap(predictions, ground_truth, coordinates):
    """
    Создает интерактивную карту с heatmap ошибок.
    """
    errors = predictions != ground_truth
    error_coords = coordinates[errors]
    
    if len(error_coords) == 0:
        print("No errors to visualize!")
        return
    
    # Центр карты - средние координаты
    center_lat = np.mean(error_coords[:, 0])
    center_lon = np.mean(error_coords[:, 1])
    
    # Создаем карту
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles='OpenStreetMap'
    )
    
    # Добавляем heatmap
    HeatMap(
        error_coords.tolist(),
        radius=15,
        blur=25,
        max_zoom=13,
        gradient={
            0.0: 'blue',
            0.5: 'yellow',
            1.0: 'red'
        }
    ).add_to(m)
    
    # Сохраняем карту
    map_path = '../results/plots/error_heatmap.html'
    m.save(map_path)
    
    print(f"\n✅ Error heatmap saved to: {map_path}")
    print(f"Total errors plotted: {len(error_coords)}")
    
    return m

if 'results' in locals():
    error_map = create_error_heatmap(
        results['predictions'],
        results['ground_truth'],
        results['coordinates']
    )

## 7. Example Predictions (Success & Failure)

In [ ]:
def show_example_predictions(test_dataset, predictions, ground_truth, class_names, n_examples=8):
    """
    Показывает примеры успешных и неуспешных предсказаний.
    """
    # Находим правильные и неправильные предсказания
    correct_mask = predictions == ground_truth
    incorrect_mask = ~correct_mask
    
    correct_indices = np.where(correct_mask)[0]
    incorrect_indices = np.where(incorrect_mask)[0]
    
    # Выбираем случайные примеры
    n_per_category = n_examples // 2
    
    if len(correct_indices) >= n_per_category:
        correct_sample = np.random.choice(correct_indices, n_per_category, replace=False)
    else:
        correct_sample = correct_indices
    
    if len(incorrect_indices) >= n_per_category:
        incorrect_sample = np.random.choice(incorrect_indices, n_per_category, replace=False)
    else:
        incorrect_sample = incorrect_indices
    
    # Визуализация
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Успешные предсказания
    for i, idx in enumerate(correct_sample):
        img, label, coords = test_dataset[idx]
        pred = predictions[idx]
        
        # Денормализация изображения для отображения
        img_np = img.permute(1, 2, 0).numpy()
        img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)
        
        axes[i].imshow(img_np)
        axes[i].set_title(
            f"✅ {class_names[label]}\nPred: {class_names[pred]}",
            fontsize=11,
            color='green',
            fontweight='bold'
        )
        axes[i].axis('off')
    
    # Неуспешные предсказания
    for i, idx in enumerate(incorrect_sample, start=n_per_category):
        img, label, coords = test_dataset[idx]
        pred = predictions[idx]
        
        img_np = img.permute(1, 2, 0).numpy()
        img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        img_np = np.clip(img_np, 0, 1)
        
        axes[i].imshow(img_np)
        axes[i].set_title(
            f"❌ True: {class_names[label]}\nPred: {class_names[pred]}",
            fontsize=11,
            color='red',
            fontweight='bold'
        )
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('../results/plots/example_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

if 'results' in locals():
    show_example_predictions(
        test_dataset,
        results['predictions'],
        results['ground_truth'],
        class_names,
        n_examples=8
    )

## Summary

Все анализы и визуализации сохранены в `results/plots/` для использования в дипломе.

In [ ]:
print("\n✅ Error analysis complete!")
print("\nGenerated files:")
print("  - results/plots/confusion_matrix.png")
print("  - results/plots/error_distance_distribution.png")
print("  - results/plots/tsne_embeddings.png")
print("  - results/plots/error_heatmap.html")
print("  - results/plots/example_predictions.png")
print("  - results/classification_report.csv")